# Prompt A/B Testing — Compare Prompt Variants Systematically
## Version Your Prompts — Systematic A/B Comparison
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/78-prompt-ab-testing/prompt_ab_testing_workbook.ipynb)

Runs the same 5 questions through two prompt variants (concise vs thorough) and scores responses by word count / 100, capped at 1.0. Identifies which variant wins per question and overall.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why prompt wording changes output dramatically; scoring strategies |
| 2 | **Variants** — VARIANT_A = concise one-sentence; VARIANT_B = thorough explanation |
| 3 | **score_response()** — word_count / 100, max 1.0 — proxy for depth/completeness |
| 4 | **Run Both Variants** — Same question through A then B in one LangGraph pass |
| 5 | **Winner Analysis** — Per-question winner + overall stats |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 1 — What Is Prompt A/B Testing?

### The core idea

A/B testing a prompt means running two versions of a system prompt on the same set of inputs and measuring which produces better outputs. Everything else stays constant:

- Same model
- Same temperature (use `0` to remove sampling noise)
- Same questions
- Same scoring function

The only variable is the prompt. This lets you attribute any difference in scores directly to the prompt change.

### Why it matters

Small prompt changes can cause large swings in:
- Response length and completeness
- Factual accuracy
- Tone and format
- Task completion rate on complex tasks

A/B testing makes those swings visible and measurable before you push a prompt change to production.

### Scoring signals

| Signal | Measures | Works when |
|--------|----------|------------|
| **Word count proxy** | Verbosity | You want to control length |
| **Semantic similarity** | Closeness to ground truth | You have expected answers |
| **LLM-as-judge** | Quality with reasoning | Open-ended responses |
| **Task completion** | Whether the task was done | You can define success programmatically |

You'll often combine multiple signals — word count alone doesn't tell you if the answer is *correct*.

> **Prerequisite:** Familiar with LangGraph StateGraph and LLM-as-judge pattern (examples 62, 76).

In [ ]:
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

# temperature=0 removes sampling noise — variation should come from the prompt, not randomness
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# the only variable in the test: two system prompt styles
VARIANT_A = "You are a concise assistant. Answer in exactly one short sentence: {question}"
VARIANT_B = "You are a thorough assistant. Provide a detailed, informative answer: {question}"

TEST_INPUTS = [
    "What is photosynthesis?",
    "How does the internet work?",
    "What is quantum entanglement?",
    "Why is the sky blue?",
    "What is the difference between RAM and storage?",
]

def ask(prompt_template: str, question: str) -> str:
    return llm.invoke([HumanMessage(content=prompt_template.format(question=question))]).content.strip()

# word-count proxy: depth correlates with word count for factual Q&A
# replace with semantic similarity or LLM-judge for production use
def word_count_score(text: str) -> float:
    return round(min(len(text.split()) / 100, 1.0), 4)

# quick sanity check on one question
q = TEST_INPUTS[0]
a, b = ask(VARIANT_A, q), ask(VARIANT_B, q)
print(f"Q: {q}")
print(f"A: {a}")
print(f"B: {b}")
print(f"Score A: {word_count_score(a):.4f}  Score B: {word_count_score(b):.4f}")

## Part 2 — LangGraph A/B Pipeline

Running both variants inside a single LangGraph invocation gives you:
- One state dict per question with both responses and scores
- Easy extension to more variants (just add more nodes)
- Optional parallel execution by routing `START → A` and `START → B` simultaneously

The sequential design (A → B) is simpler to start; switch to parallel branches once the pattern is clear.

### State shape

```python
class ABState(TypedDict):
    question:   str
    response_a: str    # filled by variant_a node
    response_b: str    # filled by variant_b node
    score_a:    float  # filled by variant_a node
    score_b:    float  # filled by variant_b node
```

### Parallel branch pattern (optional extension)

```
START → variant_a ─┐
START → variant_b ─┴→ aggregate → END
```

Add `g.add_edge(START, "variant_a")` and `g.add_edge(START, "variant_b")` — LangGraph runs them in parallel automatically.

In [ ]:
class ABState(TypedDict):
    question:   str
    response_a: str
    response_b: str
    score_a:    float
    score_b:    float

def variant_a_node(state: ABState) -> dict:
    text = ask(VARIANT_A, state["question"])
    return {"response_a": text, "score_a": word_count_score(text)}

def variant_b_node(state: ABState) -> dict:
    text = ask(VARIANT_B, state["question"])
    return {"response_b": text, "score_b": word_count_score(text)}

g = StateGraph(ABState)
g.add_node("variant_a", variant_a_node)
g.add_node("variant_b", variant_b_node)
g.add_edge(START, "variant_a")
g.add_edge("variant_a", "variant_b")
g.add_edge("variant_b", END)
ab_app = g.compile()

# run the pipeline
results = []
print(f"{'Q':<40} {'ScoreA':>7} {'ScoreB':>7} {'Winner':>7}")
print("-" * 65)
for q in TEST_INPUTS:
    r = ab_app.invoke({"question": q, "response_a": "", "response_b": "",
                       "score_a": 0.0, "score_b": 0.0})
    results.append(r)
    winner = "A" if r["score_a"] >= r["score_b"] else "B"
    print(f"{q[:39]:<40} {r['score_a']:>7.4f} {r['score_b']:>7.4f} {winner:>7}")

a_wins = sum(1 for r in results if r["score_a"] >= r["score_b"])
print(f"\nA wins: {a_wins}/{len(results)}  B wins: {len(results)-a_wins}/{len(results)}")
print(f"Avg A: {sum(r['score_a'] for r in results)/len(results):.4f}  "
      f"Avg B: {sum(r['score_b'] for r in results)/len(results):.4f}")

## Part 3 — Semantic Similarity Scoring

Word count tells you *how much* was said, not *how accurate* it was. For factual questions, semantic similarity against a reference answer is far more meaningful.

The pattern:
1. Prepare a reference answer (golden answer) per question
2. Embed both the LLM response and the reference
3. Compute cosine similarity
4. Score A and B against the *same* reference to make them comparable

### Why cosine similarity works for this

The embedding model maps semantically similar text to nearby vectors. A response that says "the sky appears blue due to Rayleigh scattering" and one that says "Rayleigh scattering causes the sky's blue color" will score high similarity even though the word order differs entirely.

In [ ]:
import numpy as np
from langchain_openai import OpenAIEmbeddings

embed = OpenAIEmbeddings(model="text-embedding-3-small")

GOLDEN_ANSWERS = {
    "What is photosynthesis?":
        "Photosynthesis converts sunlight, CO2, and water into glucose and oxygen in plant cells.",
    "How does the internet work?":
        "The internet routes data packets between devices using TCP/IP protocols across interconnected networks.",
    "What is quantum entanglement?":
        "Quantum entanglement links two particles so measuring one instantly determines the state of the other.",
    "Why is the sky blue?":
        "Rayleigh scattering causes shorter blue wavelengths to scatter more than longer wavelengths.",
    "What is the difference between RAM and storage?":
        "RAM is fast temporary memory for active processes; storage holds data persistently at lower speed.",
}

def semantic_score(response: str, reference: str) -> float:
    rv = np.array(embed.embed_query(response))
    gv = np.array(embed.embed_query(reference))
    return float(np.dot(rv, gv) / (np.linalg.norm(rv) * np.linalg.norm(gv)))

print(f"{'Q':<40} {'SimA':>6} {'SimB':>6} {'Winner':>7}")
print("-" * 64)
sem_results = []
for r in results:
    ref  = GOLDEN_ANSWERS[r["question"]]
    sa   = semantic_score(r["response_a"], ref)
    sb   = semantic_score(r["response_b"], ref)
    sem_results.append({"question": r["question"], "sim_a": sa, "sim_b": sb})
    winner = "A" if sa >= sb else "B"
    print(f"{r['question'][:39]:<40} {sa:>6.4f} {sb:>6.4f} {winner:>7}")

avg_sa = sum(r["sim_a"] for r in sem_results) / len(sem_results)
avg_sb = sum(r["sim_b"] for r in sem_results) / len(sem_results)
print(f"\nAvg semantic — A: {avg_sa:.4f}  B: {avg_sb:.4f}")
print(f"Better variant by semantic score: {'A' if avg_sa >= avg_sb else 'B'}")

## Part 4 — Multi-Variant Testing and Statistical Significance

Two variants is the minimum. In practice you may test three or more (prompt wording, few-shot examples, tone). The pipeline generalizes naturally.

### Statistical significance

With 5 test inputs, any difference could be noise. To trust a result, you need:
- At minimum 20–30 inputs per comparison
- A significance test (chi-squared for win/loss, t-test for continuous scores)

A rough rule: **if the score difference is less than 0.05 and you have fewer than 20 inputs, flip a coin** — the result isn't reliable yet.

The t-test from `scipy.stats` gives you a p-value. Common threshold: `p < 0.05` means the difference is likely real (not just noise).

In [ ]:
# Multi-variant: add a third "chain-of-thought" variant
VARIANT_C = (
    "You are an analytical assistant. First think step by step, then give a clear answer: {question}"
)

class ABCState(TypedDict):
    question:   str
    response_a: str; response_b: str; response_c: str
    score_a:    float; score_b:    float; score_c:    float

def variant_c_node(state: ABCState) -> dict:
    text = ask(VARIANT_C, state["question"])
    return {"response_c": text, "score_c": word_count_score(text)}

# extend the graph with variant C
g3 = StateGraph(ABCState)
for name, fn in [("va", variant_a_node), ("vb", variant_b_node), ("vc", variant_c_node)]:
    g3.add_node(name, fn)
g3.add_edge(START, "va"); g3.add_edge("va", "vb")
g3.add_edge("vb", "vc"); g3.add_edge("vc", END)

# for ABCState we need to map from ABState fields — re-run fresh
abc_app = StateGraph(ABCState)
abc_app.add_node("variant_a", lambda s: {"response_a": ask(VARIANT_A, s["question"]),
                                          "score_a": word_count_score(ask(VARIANT_A, s["question"]))})
abc_app.add_node("variant_b", lambda s: {"response_b": ask(VARIANT_B, s["question"]),
                                          "score_b": word_count_score(ask(VARIANT_B, s["question"]))})
abc_app.add_node("variant_c", lambda s: {"response_c": ask(VARIANT_C, s["question"]),
                                          "score_c": word_count_score(ask(VARIANT_C, s["question"]))})
abc_app.add_edge(START, "variant_a"); abc_app.add_edge("variant_a", "variant_b")
abc_app.add_edge("variant_b", "variant_c"); abc_app.add_edge("variant_c", END)
abc_compiled = abc_app.compile()

scores = {"A": [], "B": [], "C": []}
for q in TEST_INPUTS:
    r = abc_compiled.invoke({"question": q,
                              "response_a": "", "response_b": "", "response_c": "",
                              "score_a": 0.0, "score_b": 0.0, "score_c": 0.0})
    scores["A"].append(r["score_a"])
    scores["B"].append(r["score_b"])
    scores["C"].append(r["score_c"])

print(f"{'Variant':<10} {'Mean':>8} {'Min':>8} {'Max':>8}")
print("-" * 40)
for v, vals in scores.items():
    print(f"{v:<10} {sum(vals)/len(vals):>8.4f} {min(vals):>8.4f} {max(vals):>8.4f}")

# p-value significance check between best two variants
try:
    from scipy import stats
    best_v = max(scores, key=lambda v: sum(scores[v]))
    others = [v for v in scores if v != best_v]
    for other in others:
        t, p = stats.ttest_rel(scores[best_v], scores[other])
        print(f"\n{best_v} vs {other}: t={t:.3f}  p={p:.4f}  "
              f"({'significant' if p < 0.05 else 'not significant yet — need more data'})")
except ImportError:
    print("\nInstall scipy for significance testing: pip install scipy")

## Exercises

---

**Exercise 1 — Add a Few-Shot Variant**

Create `VARIANT_D` that includes two example Q&A pairs in the prompt (few-shot prompting). Run it against the golden answers and compare the semantic similarity score to Variants A and B.

---

**Exercise 2 — LLM-as-Judge Scoring**

Replace `word_count_score()` with an LLM-as-judge scorer (from example 76). The judge should rate how well each response answers the question on a 0–2 scale. Which variant wins under this metric?

---

**Exercise 3 — Swap Model Between Variants**

Instead of swapping prompts, swap models: `gpt-4o-mini` (Variant A) vs a hypothetical cheaper/faster model. Show that the pipeline structure stays identical — only the `llm` object changes per node.

---

**Exercise 4 — Bootstrap Confidence Intervals**

With only 5 samples, use bootstrap resampling to estimate a 95% confidence interval for the mean score of each variant. If the confidence intervals overlap, the difference is not reliable.

*Hint:* Sample with replacement 1000 times using `random.choices(scores, k=len(scores))`.

---
## Answer Key

Attempt the exercises before reading below.

In [ ]:
# Answer 1 — Few-Shot Variant
VARIANT_D = '''You are a helpful assistant. Here are two examples:

Q: What is DNA?
A: DNA is a double-helix molecule that stores genetic instructions in sequences of nucleotide bases.

Q: What is gravity?
A: Gravity is a fundamental force that attracts objects with mass toward each other.

Now answer this question in a similar style:
Q: {question}
A:'''

print(f"{'Q':<40} {'SimA':>6} {'SimB':>6} {'SimD':>6} {'Best':>6}")
print("-" * 64)
for r in results:
    ref   = GOLDEN_ANSWERS[r["question"]]
    resp_d = ask(VARIANT_D, r["question"])
    sa    = semantic_score(r["response_a"], ref)
    sb    = semantic_score(r["response_b"], ref)
    sd    = semantic_score(resp_d, ref)
    best  = max([("A", sa), ("B", sb), ("D", sd)], key=lambda x: x[1])[0]
    print(f"{r['question'][:39]:<40} {sa:>6.4f} {sb:>6.4f} {sd:>6.4f} {best:>6}")

In [ ]:
# Answer 4 — Bootstrap Confidence Intervals
import random

def bootstrap_ci(values: list[float], n_boot: int = 1000, ci: float = 0.95) -> tuple[float, float]:
    means = [sum(random.choices(values, k=len(values))) / len(values) for _ in range(n_boot)]
    means.sort()
    lo = int((1 - ci) / 2 * n_boot)
    hi = int((1 + ci) / 2 * n_boot)
    return means[lo], means[hi]

print("Bootstrap 95% CIs (word-count scores):")
for v, vals in scores.items():
    lo, hi = bootstrap_ci(vals)
    mean   = sum(vals) / len(vals)
    print(f"  Variant {v}: mean={mean:.4f}  CI=[{lo:.4f}, {hi:.4f}]")

print()
print("Note: overlapping CIs → difference is not statistically reliable with this sample size.")
print("Add more test questions (aim for 30+) to get reliable separation.")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*